### Configuration

In [1]:
# --- CONFIGURATION AUTOMATIQUE ---
%load_ext autoreload
%autoreload 2

### Imports

In [1]:
# --- IMPORTS SYSTÈME & CHEMINS ---
import os
import sys
import logging
import warnings
import json
# --- MANIPULATION DE DONNÉES & VISUALISATION ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- MLOPS (MLflow) ---
import mlflow
import mlflow.sklearn

# --- SCIKIT-LEARN & MODÈLES ---
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    make_scorer, 
    roc_auc_score, 
    balanced_accuracy_score, 
    ConfusionMatrixDisplay, 
    roc_curve, 
    auc
)

### import des fonctions crées

In [ ]:
# MODULES SOURCES
from src.test_separation_fonctions import pipeline_model, find_best_threshold
from src.model_utils import load_data, custom_business_cost

In [ ]:
# Ajout du dossier racine au sys.path pour détecter le dossier /src
# On remonte d'un cran car le notebook est dans /notebooks
sys.path.append(os.path.abspath('..'))

### Configuration de l'affichage

In [ ]:
%matplotlib inline
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

### Chargement des données

In [ ]:
X_full, y_full = load_data()

### Fractionnement des données 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, train_size=50000, stratify=y_full, random_state=42
)

print(f"Dataset prêt pour l'entraînement : {X_train.shape}")

### Configuration de l'expérience globale dans MLflow

In [ ]:
mlflow.set_experiment("Credit_Scoring_Training_Full")

### Initialisation des paramètres de validation croisée et de la métrique métier

In [ ]:
biz_scorer = make_scorer(custom_business_cost, greater_is_better=False, needs_proba=True)
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring_metrics = {'AUC': 'roc_auc', 'Cost': biz_scorer}

### définitions de mes modèles à tester

In [ ]:
model_types = ["Logistic_Regression", "XGBoost", "LightGBM"]

### Entrainement et optimisation

In [ ]:
for m_type in model_types:
    pipeline, param_grid = pipeline_model(m_type)
    
    with mlflow.start_run(run_name=f"Optimization_{m_type}"):
        mlflow.sklearn.autolog(log_models=True, log_datasets=True, log_input_examples=True, disable=False)
        
        # 1. Phase d'entraînement (GridSearch)
        grid = GridSearchCV(pipeline, param_grid, cv=cv_strategy, 
                            scoring=scoring_metrics, refit='Cost', n_jobs=-1)
        grid.fit(X_train, y_train)
        
        # 2. Phase d'optimisation métier (Seuil)
        
        y_proba = grid.best_estimator_.predict_proba(X_train)[:, 1]
        best_thresh, min_cost = find_best_threshold(y_train, y_proba)
        y_pred_optimal = (y_proba >= best_thresh).astype(int)

        mlflow.log_metric("optimized_threshold", best_thresh)
        mlflow.log_metric("min_business_cost", min_cost)
        mlflow.log_metric("final_balanced_accuracy", balanced_accuracy_score(y_train, y_pred_optimal))

        # Génération de l'artefact Confusion Matrix
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(y_train, y_pred_optimal, cmap='Blues', ax=ax)
        ax.set_title(f"Confusion Matrix: {m_type}\n(Threshold: {best_thresh:.2f})")
        
        cm_path = f"cm_{m_type}.png"
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.show()

        print(f"✅ Modèle {m_type} optimisé.")

### Registry

In [ ]:
# On récupère les résultats de l'expérience pour choisir les champions
current_experiment = mlflow.get_experiment_by_name("Credit_Scoring_Training_Full")
runs = mlflow.search_runs(experiment_ids=[current_experiment.experiment_id])

for m_type in model_types:
    # On trouve le meilleur run pour ce modèle précis
    best_run = runs[runs['tags.mlflow.runName'] == f"Optimization_{m_type}"].iloc[0]
    run_id = best_run['run_id']
    
    # Enregistrement officiel
    model_uri = f"runs:/{run_id}/model"
    mlflow.register_model(model_uri, f"Champion_{m_type}")
    
    print(f"🏆 {m_type} enregistré dans le Model Registry.")